# 03_validation: Cross-Disease Validation and Comparison

This notebook validates results against expected regions
and compares across diseases.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from validation import load_expected_regions, compare_gene_lists
from report import generate_html_report

In [ ]:
# Load expected regions
expected = load_expected_regions()
print(expected.to_string())

In [ ]:
# Load all results
diseases = ["Parkinson", "Alzheimer", "Huntington"]
list_types = ["strict", "broad"]

all_results = {}
for disease in diseases:
    for lt in list_types:
        try:
            stats = pd.read_parquet(f"../results/stats_{disease.lower()}_{lt}.parquet")
            validation = pd.read_parquet(f"../results/validation_{disease.lower()}_{lt}.parquet")
            all_results[f"{disease}_{lt}"] = {"stats": stats, "validation": validation}
            print(f"Loaded {disease} ({lt}): {len(stats)} regions")
        except FileNotFoundError:
            print(f"Not found: {disease} ({lt})")

In [ ]:
# Compare strict vs broad for each disease
for disease in diseases:
    try:
        strict = pd.read_parquet(f"../results/stats_{disease.lower()}_strict.parquet")
        broad = pd.read_parquet(f"../results/stats_{disease.lower()}_broad.parquet")
        comparison = compare_gene_lists(strict, broad)
        print(f"\n{disease} agreement:")
        print(comparison['agreement'].value_counts())
    except FileNotFoundError:
        print(f"Results not found for {disease}")

In [ ]:
# Cross-disease correlation (strict lists only)
strict_results = {}
for disease in diseases:
    try:
        stats = pd.read_parquet(f"../results/stats_{disease.lower()}_strict.parquet")
        strict_results[disease] = stats["score"]
    except FileNotFoundError:
        pass

if len(strict_results) > 1:
    score_df = pd.DataFrame(strict_results)
    print("Score correlation matrix:")
    print(score_df.corr())

    # Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(score_df.corr(), annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
    plt.title("Cross-Disease Score Correlation (Strict Lists)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Badge summary across diseases
badge_summary = []
for disease in diseases:
    try:
        val = pd.read_parquet(f"../results/validation_{disease.lower()}_strict.parquet")
        counts = val["badge"].value_counts()
        for badge, count in counts.items():
            badge_summary.append({"disease": disease, "badge": badge, "count": count})
    except FileNotFoundError:
        pass

if badge_summary:
    badge_df = pd.DataFrame(badge_summary)
    pivot = badge_df.pivot(index="disease", columns="badge", values="count").fillna(0).astype(int)
    print(pivot)

    pivot.plot(kind='bar', stacked=True, figsize=(10, 6))
    plt.title("Evidence Badges by Disease")
    plt.ylabel("Number of Regions")
    plt.tight_layout()
    plt.show()

In [ ]:
# Generate HTML reports
for disease in diseases:
    for lt in list_types:
        try:
            report_path = generate_html_report(disease, lt)
            print(f"Generated: {report_path}")
        except FileNotFoundError:
            print(f"Results not found for {disease} ({lt})")

## Validation Checklist

For each disease, verify:

1. [ ] Expected regions match literature
2. [ ] Supported badges align with known affected regions
3. [ ] Mismatches documented as findings (not forced)
4. [ ] Negative control does not flag brain regions
5. [ ] Strict vs broad lists show reasonable agreement
6. [ ] Donor consistency >= 4/6 for top regions

## Document All Mismatches

If a known affected region is NOT supported:
- Check if region exists in atlas (e.g., substantia nigra missing from Desikan-Killiany)
- Check gene list coverage
- Check mapping method
- Report as finding: "Region X expected but not supported - possible reasons: ..."